# Chatbot Latinoamérica Comparte — Semana 4
## Pipeline Completo + Interfaz Gradio

**Electiva Machine Learning | Ingeniería de Sistemas — USTA Tunja**  
**Equipo:** Giovanni Torres ,  Mateo Gómez

---

### Arquitectura del pipeline

```
Usuario  ──>  Interfaz Gradio
                    |
             chatbot_rag(query)
                    |
            retrieve_context()  ──>  FAISS + Corpus
                    |
            generate_answer()   ──>  Qwen LLM
                    |
            Respuesta  ──>  Interfaz Gradio
```


In [1]:
# ================================================================
# CELDA 0 — Generar corpus y embeddings (ejecutar en sesion nueva)
# ================================================================
!pip install -q sentence-transformers faiss-cpu gradio transformers accelerate pandas numpy

import os, re, json, pickle
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

os.makedirs('data', exist_ok=True)

# Corpus
corpus = [
    {"intent": "saludo",
     "utterances": ["Hola","Buenos días","Buenas tardes, ¿me pueden ayudar?",
                    "Hey, ¿hay alguien ahí?","Qué tal",
                    "Hola, quiero saber más de Latinoamérica Comparte","Saludos","¿Cómo están?"],
     "respuesta": "¡Hola! Bienvenido a Latinoamérica Comparte. ¿En qué te puedo ayudar? Puedo contarte sobre Comparte Academia, Comparte Liderazgo o Comparte Talento."},

    {"intent": "que_es_latinoamerica_comparte",
     "utterances": ["¿Qué es Latinoamérica Comparte?","¿De qué se trata esta organización?",
                    "Cuéntame sobre Colombia Comparte","¿A qué se dedican?",
                    "Vi su página y no entendí bien qué hacen","¿Es una empresa, una fundación o qué?",
                    "Háblame de Latinoamérica Comparte","¿Qué hace este ecosistema?"],
     "respuesta": "Latinoamérica Comparte es un ecosistema de emprendimiento, liderazgo y talento. Operamos a través de tres líneas: Comparte Academia (formación), Comparte Liderazgo (desarrollo humano) y Comparte Talento (speakers y eventos)."},

    {"intent": "programa_deskubre",
     "utterances": ["¿Qué es DESKUBRE?","Me quedé sin trabajo y quiero emprender",
                    "Tengo una idea pero no sé cómo arrancar","Quiero montar negocio pero no sé nada",
                    "¿Tienen algo para principiantes en emprendimiento?","DESKUBRE",
                    "¿Cuánto dura el programa inicial de emprendimiento?",
                    "Quiero descubrir si sirvo para emprender"],
     "respuesta": "DESKUBRE es nuestro programa inicial de emprendimiento de 1 mes. Trabajarás mentalidad emprendedora, validación de idea y orientación. Perfecto si estás empezando. Hace parte de Comparte Academia."},

    {"intent": "programa_estructura",
     "utterances": ["¿Qué es ESTRUCTURA?","Ya tengo mi negocio pero necesito organizarlo mejor",
                    "Tengo un emprendimiento pero no está generando lo que quiero","EDIFICA ESTRUCTURA",
                    "¿Cuánto dura el programa de emprendimiento avanzado?",
                    "Quiero escalar mi negocio, ¿qué programa me recomiendan?",
                    "¿Tienen algo de finanzas, marketing y ventas para emprendedores?",
                    "Mi empresa lleva 2 años y quiero fortalecerla"],
     "respuesta": "ESTRUCTURA es nuestro programa avanzado de 12 meses para emprendedores con negocio en marcha. Cubre modelo de negocio, finanzas, marketing, ventas, liderazgo y acompañamiento personalizado."},

    {"intent": "comparte_academia",
     "utterances": ["¿Qué es Comparte Academia?","¿Qué programas tienen para emprendedores?",
                    "Quiero formarme en emprendimiento","¿Tienen cursos de negocios?","EDIFICA",
                    "¿Dónde puedo aprender a emprender con ustedes?",
                    "¿Cuáles son sus programas de formación?",
                    "Quiero crecer como emprendedor, ¿qué me ofrecen?"],
     "respuesta": "Comparte Academia es nuestra línea de formación. Incluye DESKUBRE (1 mes, para empezar) y ESTRUCTURA (12 meses, para escalar). El objetivo es acompañarte desde la idea hasta el vuelo."},

    {"intent": "comparte_liderazgo",
     "utterances": ["¿Qué es Comparte Liderazgo?","Soy líder en una empresa y quiero desarrollar a mi equipo",
                    "¿Tienen programas de cultura organizacional?",
                    "Necesito transformar el liderazgo en mi empresa","¿Ofrecen talleres para empresas?",
                    "¿Cómo puedo llevar el pensamiento emprendedor a mi organización?",
                    "Quiero mejorar el desarrollo humano en mi equipo",
                    "¿Tienen formación para directivos o gerentes?"],
     "respuesta": "Comparte Liderazgo está enfocada en liderazgo, desarrollo humano y cultura organizacional para empresas. Llevamos el pensamiento emprendedor al interior de las organizaciones."},

    {"intent": "comparte_talento",
     "utterances": ["¿Qué es Comparte Talento?","Necesito un conferencista para un evento de mi empresa",
                    "¿Tienen speakers disponibles?","Top Speakers","¿Hacen eventos corporativos?",
                    "Quiero contratar una conferencia motivacional",
                    "¿Tienen artistas o experiencias para eventos empresariales?",
                    "Organizo eventos y busco conferencistas de alto nivel"],
     "respuesta": "Comparte Talento (antes Top Speakers) es nuestra línea de conferencias, speakers, artistas y eventos corporativos. Contamos con talento seleccionado para transformar audiencias empresariales."},

    {"intent": "costos_precios",
     "utterances": ["¿Cuánto cuesta inscribirse?","¿Cuál es el precio de DESKUBRE?",
                    "¿Tienen financiación o facilidades de pago?","¿Cuánto vale ESTRUCTURA?",
                    "¿Es muy caro?","No tengo mucha plata, ¿hay becas?",
                    "¿Qué incluye el costo del programa?","¿Cuánto tengo que pagar para entrar?"],
     "respuesta": "Los costos varían según el programa. Te recomendamos contactarnos directamente en colombiacomparte.com para recibir información actualizada sobre precios y planes de financiación."},

    {"intent": "inscripcion",
     "utterances": ["¿Cómo me inscribo?","Quiero registrarme en DESKUBRE",
                    "¿Cuáles son los requisitos para entrar?","¿Dónde me anoto?",
                    "Quiero empezar ya, ¿qué hago?","¿Hay fechas de inicio?",
                    "¿Cómo es el proceso para unirme?","Quiero inscribirme ahora mismo"],
     "respuesta": "Para inscribirte visita colombiacomparte.com o escríbenos por nuestros canales. Un asesor te guiará en el proceso y te indicará las fechas disponibles."},

    {"intent": "despedida",
     "utterances": ["Gracias, hasta luego","Chao","Eso era todo, gracias",
                    "Listo, ya resolví mi duda","Hasta pronto","Bueno, muchas gracias",
                    "Me voy, gracias por la info","Adiós"],
     "respuesta": "Fue un placer atenderte. En Latinoamérica Comparte estamos para acompañarte. Si tienes más preguntas, aquí estaremos."},
]

rows = []
for item in corpus:
    for utt in item["utterances"]:
        rows.append({"intent": item["intent"], "utterance": utt, "respuesta": item["respuesta"]})

df = pd.DataFrame(rows)

def limpiar_texto(texto):
    texto = texto.lower().strip()
    texto = re.sub(r'[^a-záéíóúüñ0-9\s¿?¡!.,]', ' ', texto)
    return re.sub(r'\s+', ' ', texto)

df["utterance_clean"] = df["utterance"].apply(limpiar_texto)

print("Generando embeddings...")
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = modelo_embeddings.encode(
    df["utterance_clean"].tolist(),
    show_progress_bar=True,
    normalize_embeddings=True
)

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings.astype(np.float32))

df.to_csv('data/corpus.csv', index=False)
faiss.write_index(index, 'data/corpus_index.faiss')
with open('data/corpus_raw.json', 'w', encoding='utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)

print(f"Dataset listo: {len(df)} utterances | Indice FAISS: {index.ntotal} vectores")
print(f"Intents: {df['intent'].nunique()} | Dimension: {embeddings.shape[1]}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 45.4 MB/s eta 0:00:00
Generando embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Dataset listo: 80 utterances | Indice FAISS: 80 vectores
Intents: 10 | Dimension: 384


## 1. Instalación de dependencias


In [2]:
!pip install -q gradio transformers accelerate sentence-transformers faiss-cpu pandas numpy


## 2. Carga de componentes del pipeline


In [3]:
import pandas as pd
import numpy as np
import faiss
import re
import os
import json
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Librerias importadas | Dispositivo: {device}")


Librerias importadas | Dispositivo: cpu


In [4]:
# Ajustar la ruta si se clona un repositorio
# !git clone https://github.com/TU_USUARIO/TU_REPO.git
# os.chdir('TU_REPO')

assert os.path.exists('data/corpus.csv'),         "Ejecutar la celda de setup primero"
assert os.path.exists('data/corpus_index.faiss'), "Ejecutar la celda de setup primero"

df    = pd.read_csv('data/corpus.csv')
index = faiss.read_index('data/corpus_index.faiss')

print("Cargando modelos...")
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
modelo_llm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
if device == "cpu":
    modelo_llm = modelo_llm.to(device)
modelo_llm.eval()

print("Todos los modelos cargados")


Cargando modelos...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Todos los modelos cargados


## 3. Pipeline completo


In [5]:
# Configuración
UMBRAL_CONFIANZA  = 0.50
TOP_K             = 3
GEN_PARAMS = {
    "max_new_tokens":      200,
    "temperature":         0.2,
    "top_p":               0.85,
    "do_sample":           True,
    "repetition_penalty":  1.1,
}
FALLBACK_MSG = (
    "Lo siento, no tengo información sobre eso en mis datos actuales. "
    "Para más información visita colombiacomparte.com o escríbenos "
    "por nuestros canales oficiales."
)

# Funciones del pipeline
def limpiar_texto(texto):
    texto = texto.lower().strip()
    texto = re.sub(r'[^a-záéíóúüñ0-9\s¿?¡!.,]', ' ', texto)
    return re.sub(r'\s+', ' ', texto)


def retrieve_context(query):
    q_emb = modelo_embeddings.encode(
        [limpiar_texto(query)], normalize_embeddings=True
    ).astype(np.float32)
    scores, idxs  = index.search(q_emb, TOP_K)
    best_score    = float(scores[0][0])
    best_idx      = int(idxs[0][0])
    if best_score < UMBRAL_CONFIANZA:
        return {"found": False, "score": round(best_score, 4),
                "intent": "desconocido", "respuesta": FALLBACK_MSG}
    return {"found": True,  "score": round(best_score, 4),
            "intent": df["intent"].iloc[best_idx],
            "respuesta": df["respuesta"].iloc[best_idx]}


def construir_prompt(query, contexto):
    return [
        {"role": "system", "content": (
            "Eres el asistente virtual de Latinoamérica Comparte. "
            "Responde SOLO con la información del CONTEXTO. "
            "Nunca inventes datos. Sé claro, amigable y en español."
        )},
        {"role": "user",
         "content": f"CONTEXTO:\n{contexto}\n\nPREGUNTA: {query}\n\nRespuesta:"},
    ]


def generate_answer(query, context):
    messages   = construir_prompt(query, context)
    prompt_txt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        prompt_txt, return_tensors="pt", truncation=True, max_length=1024
    ).to(device)
    with torch.no_grad():
        out_ids = modelo_llm.generate(
            **inputs, **GEN_PARAMS,
            pad_token_id=tokenizer.eos_token_id
        )
    new_tokens = out_ids[0][inputs["input_ids"].shape[1]:]
    respuesta  = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    for tok in ["<|im_end|>", "<|im_start|>", "assistant", "Assistant"]:
        respuesta = respuesta.replace(tok, "").strip()
    return respuesta if len(respuesta) >= 20 else context


def chatbot_rag(query):
    resultado = retrieve_context(query)
    if not resultado["found"]:
        return resultado["respuesta"], "desconocido", 0.0
    respuesta = generate_answer(query, resultado["respuesta"])
    return respuesta, resultado["intent"], resultado["score"]


print("Pipeline definido")


Pipeline definido


## 4. Interfaz Gradio

Interfaz de chat con historial, indicador de intent y score de confianza.


In [6]:
import gradio as gr

EJEMPLOS = [
    "¿Qué es Latinoamérica Comparte?",
    "Quiero emprender pero no sé cómo empezar",
    "Ya tengo mi negocio y quiero fortalecerlo",
    "Necesito un speaker para un evento corporativo",
    "¿Cuánto cuesta inscribirse?",
    "¿Cómo me inscribo en DESKUBRE?",
]

CSS = """
#chatbot-container { font-family: 'Segoe UI', Arial, sans-serif; }
.message.bot { background: #f0f4f8 !important; border-left: 3px solid #1a56db; }
.message.user { background: #e8f5e9 !important; }
#info-panel { font-size: 0.82rem; color: #555; padding: 6px 0; }
#send-btn { background: #1a56db !important; }
"""

def responder(mensaje_usuario, historial):
    if not mensaje_usuario.strip():
        return historial, historial, "", ""
    respuesta, intent, score = chatbot_rag(mensaje_usuario)
    historial = historial + [[mensaje_usuario, respuesta]]
    info = f"Intent detectado: **{intent}** | Confianza: **{score:.2f}**"
    return historial, historial, "", info


def limpiar_chat():
    return [], [], "", ""


with gr.Blocks(
    title="Chatbot Latinoamérica Comparte",
    theme=gr.themes.Soft(primary_hue="blue", secondary_hue="cyan"),
    css=CSS,
    elem_id="chatbot-container",
) as demo:

    gr.Markdown("""
    # Chatbot — Latinoamérica Comparte
    Asistente virtual con arquitectura RAG · USTA Tunja · Electiva Machine Learning
    """)

    with gr.Row():
        with gr.Column(scale=3):
            chatbot_ui = gr.Chatbot(
                label="Conversación",
                height=460,
                bubble_full_width=False,
                show_label=False,
            )
            with gr.Row():
                txt_input = gr.Textbox(
                    label="",
                    placeholder="Escribe tu pregunta aquí...",
                    scale=5,
                    lines=1,
                    show_label=False,
                )
                btn_enviar = gr.Button("Enviar", variant="primary", scale=1, elem_id="send-btn")

            with gr.Row():
                btn_limpiar = gr.Button("Limpiar conversacion", variant="secondary", size="sm")

            info_box = gr.Markdown("", elem_id="info-panel")

        with gr.Column(scale=1):
            gr.Markdown("**Preguntas de ejemplo**")
            for ejemplo in EJEMPLOS:
                gr.Button(ejemplo, size="sm").click(
                    fn=lambda e=ejemplo, h=[]: responder(e, h),
                    inputs=[gr.State(ejemplo), gr.State([])],
                    outputs=[chatbot_ui, gr.State(), txt_input, info_box],
                )

            gr.Markdown("""
            ---
            **Arquitectura**
            - Embeddings: paraphrase-multilingual-MiniLM-L12-v2
            - Busqueda: FAISS IndexFlatIP
            - LLM: Qwen2.5-0.5B-Instruct
            - Intents: 10 categorias
            - Corpus: 80 utterances
            """)

    state = gr.State([])

    btn_enviar.click(
        fn=responder,
        inputs=[txt_input, state],
        outputs=[chatbot_ui, state, txt_input, info_box],
    )
    txt_input.submit(
        fn=responder,
        inputs=[txt_input, state],
        outputs=[chatbot_ui, state, txt_input, info_box],
    )
    btn_limpiar.click(
        fn=limpiar_chat,
        outputs=[chatbot_ui, state, txt_input, info_box],
    )

print("Interfaz Gradio definida")


/tmp/ipykernel_4991/1101572834.py:33: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_4991/1101572834.py:33: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_4991/1101572834.py:47: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(
/tmp/ipykernel_4991/1101572834.py:47: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot_ui = gr.Chatbot(
/tmp/ipykernel_49

Interfaz Gradio definida


## 5. Lanzar el chatbot

Ejecutar la siguiente celda para abrir el chatbot.  
En Colab, `share=True` genera un enlace público temporal para la demo.


In [7]:
demo.launch(
    share=True,
    debug=False,
    show_error=True,
)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fb56372127325ff353.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6. Evaluación y métricas

Ejecutar esta sección **sin Gradio activo** para obtener las métricas del informe.


In [8]:
eval_final = [
    # (pregunta, intent_esperado, debe_responder)
    ("Hola, buenos días",                              "saludo",                       True),
    ("¿Qué es Latinoamérica Comparte?",                "que_es_latinoamerica_comparte", True),
    ("quiero montar negocio pero no sé nada",          "programa_deskubre",             True),
    ("ya tengo empresa y quiero escalarla",             "programa_estructura",           True),
    ("necesito un speaker para un evento corporativo", "comparte_talento",              True),
    ("¿cuánto vale el programa?",                      "costos_precios",                True),
    ("¿cómo me registro?",                             "inscripcion",                   True),
    ("gracias, hasta luego",                           "despedida",                     True),
    # Fuera de dominio
    ("¿cuál es la capital de Francia?",                "desconocido",                   False),
    ("recomiéndame una receta de cocina",              "desconocido",                   False),
]

print("Evaluacion final del chatbot")
print()
print(f"{'#':>2}  {'Pregunta':44}  {'Intent Esp.':24}  {'Intent Obt.':24}  {'Score':>6}  {'OK':>3}")
print("-" * 112)

correctas = 0
for i, (query, intent_esp, debe_responder) in enumerate(eval_final, 1):
    respuesta, intent_obt, score = chatbot_rag(query)
    ok = (intent_obt == intent_esp) or (not debe_responder and intent_obt == "desconocido")
    if ok:
        correctas += 1
    marca = "OK" if ok else "--"
    print(f"{i:>2}  {query[:42]:44}  {intent_esp:24}  {intent_obt:24}  {score:>6.3f}  {marca:>3}")

precision = correctas / len(eval_final) * 100
print("-" * 112)
print(f"\nRESULTADO: {correctas}/{len(eval_final)} correctas = {precision:.1f}%")
print(f"Objetivo (>= 80%): {'CUMPLIDO' if precision >= 80 else 'No alcanzado'}")


Evaluacion final del chatbot

 #  Pregunta                                      Intent Esp.               Intent Obt.                Score   OK
----------------------------------------------------------------------------------------------------------------
 1  Hola, buenos días                             saludo                    saludo                     0.925   OK
 2  ¿Qué es Latinoamérica Comparte?               que_es_latinoamerica_comparte  que_es_latinoamerica_comparte   1.000   OK
 3  quiero montar negocio pero no sé nada         programa_deskubre         programa_deskubre          1.000   OK
 4  ya tengo empresa y quiero escalarla           programa_estructura       programa_estructura        0.724   OK
 5  necesito un speaker para un evento corpora    comparte_talento          comparte_talento           0.887   OK
 6  ¿cuánto vale el programa?                     costos_precios            costos_precios             0.704   OK
 7  ¿cómo me registro?                           

## 7. Guardar resultados


In [9]:
resultados_informe = {
    "precision_total":    round(precision, 2),
    "correctas":          correctas,
    "total":              len(eval_final),
    "objetivo_cumplido":  precision >= 80,
    "config": {
        "modelo_embeddings":  "paraphrase-multilingual-MiniLM-L12-v2",
        "modelo_llm":         "Qwen/Qwen2.5-0.5B-Instruct",
        "umbral_confianza":   UMBRAL_CONFIANZA,
        "top_k":              TOP_K,
        "total_intents":      10,
        "total_utterances":   80,
    },
}

with open('data/resultados_evaluacion.json', 'w', encoding='utf-8') as f:
    json.dump(resultados_informe, f, ensure_ascii=False, indent=2)

print("Resultados guardados en data/resultados_evaluacion.json")
print(json.dumps(resultados_informe, indent=2, ensure_ascii=False))


Resultados guardados en data/resultados_evaluacion.json
{
  "precision_total": 100.0,
  "correctas": 10,
  "total": 10,
  "objetivo_cumplido": true,
  "config": {
    "modelo_embeddings": "paraphrase-multilingual-MiniLM-L12-v2",
    "modelo_llm": "Qwen/Qwen2.5-0.5B-Instruct",
    "umbral_confianza": 0.5,
    "top_k": 3,
    "total_intents": 10,
    "total_utterances": 80
  }
}
